# Meeting Notes to Architecture

Turn a Teams meeting recap / transcript into (1) a structured summary of **discussion points, decisions and proposals**, and (2) a **high-resolution architecture diagram** of the solution that was discussed.

## Pipeline

```
Teams recap link / VTT / DOCX / pasted text
        -> transcript extraction & cleanup
        -> LLM structured extraction (summary | decisions | proposals | actions | risks)
        -> LLM architecture spec (layers, components, data flows)
        -> Graphviz render (PNG @ 300 DPI + SVG) + Mermaid export
```

## Prerequisites

| Need | How |
| --- | --- |
| Azure OpenAI | Set `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT`; auth via `AZURE_OPENAI_API_KEY` **or** Entra ID (`az login`) |
| Graph transcript access (optional) | Delegated: `OnlineMeetings.Read` + you are the organizer. App-only: `OnlineMeetingTranscript.Read.All` + [application access policy](https://learn.microsoft.com/graph/cloud-communication-online-meeting-application-access-policy) |
| Graphviz | `pip install graphviz` **and** the Graphviz binaries (`winget install Graphviz.Graphviz`), otherwise the notebook falls back to Mermaid only |

> **Secrets are never hard-coded.** All credentials are read from environment variables.
> If Graph access is not available, download the transcript from the recap page
> (**Recap -> Transcript -> Download -> .vtt/.docx**) and point `TRANSCRIPT_FILE` at it.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 ▸ Dependencies (run once per environment)
# ─────────────────────────────────────────────────────────────────────────────
%pip install -q openai azure-identity msal requests webvtt-py python-docx graphviz


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 ▸ Configuration  (no secrets in code — everything comes from env vars)
# ─────────────────────────────────────────────────────────────────────────────
import os, re, json
from pathlib import Path
from urllib.parse import urlparse, parse_qs, unquote

# ── Meeting identity ──────────────────────────────────────────────────────────
MEETING_TITLE = "S&PE / Microsoft Fabric Discussion"
MEETING_DATE  = "2026-07-29"

# Paste the Teams "Meeting recap" link here (used to derive threadId / organizerId).
MEETING_RECAP_URL = ""

# ── Transcript source ─────────────────────────────────────────────────────────
#   "graph" -> pull the transcript from Microsoft Graph using MEETING_RECAP_URL
#   "file"  -> read a downloaded .vtt / .docx / .txt transcript
#   "text"  -> use the TRANSCRIPT_INLINE string below (paste notes directly)
TRANSCRIPT_SOURCE = "file"
TRANSCRIPT_FILE   = r"./transcripts/meeting.vtt"
TRANSCRIPT_INLINE = """
Paste raw meeting notes or transcript text here when TRANSCRIPT_SOURCE = "text".
"""

# ── Azure OpenAI ──────────────────────────────────────────────────────────────
# Verified working with Entra ID (az login) — no API key required.
AOAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT", "https://aitestepm.openai.azure.com/")
AOAI_DEPLOYMENT  = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-5.2")
AOAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
AOAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")      # optional — Entra ID used when unset

# ── Microsoft Graph (only needed for TRANSCRIPT_SOURCE = "graph") ─────────────
GRAPH_TENANT_ID     = os.getenv("GRAPH_TENANT_ID", "72f988bf-86f1-41af-91ab-2d7cd011db47")
GRAPH_CLIENT_ID     = os.getenv("GRAPH_CLIENT_ID", "")     # your app registration
GRAPH_CLIENT_SECRET = os.getenv("GRAPH_CLIENT_SECRET")     # set => app-only, unset => device code

# Graphviz is installed outside PATH for kernels started before the install.
GRAPHVIZ_BIN = r"C:\Program Files\Graphviz\bin"
if os.path.isdir(GRAPHVIZ_BIN) and GRAPHVIZ_BIN not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + GRAPHVIZ_BIN

# ── Output ────────────────────────────────────────────────────────────────────
OUT_DIR = Path("./output") / f"{MEETING_DATE}_{re.sub(r'[^A-Za-z0-9]+', '-', MEETING_TITLE).strip('-')}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIAGRAM_DPI = 300          # high-resolution raster output
REDACT_NAMES = False       # True => replace speaker names with Speaker 1..N before sending to the LLM

print(f"Output folder : {OUT_DIR.resolve()}")
print(f"AOAI endpoint : {AOAI_ENDPOINT or '<not set — export AZURE_OPENAI_ENDPOINT>'}")
print(f"AOAI model    : {AOAI_DEPLOYMENT}")
print(f"AOAI auth     : {'API key' if AOAI_API_KEY else 'Entra ID (DefaultAzureCredential)'}")
print(f"Graphviz      : {'found' if __import__('shutil').which('dot') else 'NOT found — Mermaid only'}")


## Step 1 — Get the transcript

A Teams recap URL carries everything needed to locate the meeting: `threadId` (the meeting chat),
`organizerId` and `tenantId`. The next cell parses those out; the cell after it uses them against
Microsoft Graph. If Graph access is blocked, download the transcript from the recap page and switch
`TRANSCRIPT_SOURCE` to `"file"`.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 ▸ Parse a Teams meeting-recap URL
# ─────────────────────────────────────────────────────────────────────────────
def parse_recap_url(url: str) -> dict:
    """Extract meeting identifiers from a Teams `meetingrecap` deep link."""
    if not url:
        return {}
    q = parse_qs(urlparse(url).query)
    one = lambda k: unquote(q[k][0]) if k in q and q[k] else None
    info = {
        "thread_id":    one("threadId"),
        "organizer_id": one("organizerId"),
        "tenant_id":    one("tenantId"),
        "drive_id":     one("driveId"),
        "drive_item_id": one("driveItemId"),
        "file_url":     one("fileUrl"),
        "site_path":    one("sitePath"),
    }
    return {k: v for k, v in info.items() if v}


RECAP = parse_recap_url(MEETING_RECAP_URL)
if RECAP:
    for k, v in RECAP.items():
        print(f"{k:14}: {v[:110]}")
else:
    print("No recap URL supplied — using TRANSCRIPT_SOURCE =", TRANSCRIPT_SOURCE)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 ▸ Microsoft Graph — locate the meeting and download its VTT transcript
# ─────────────────────────────────────────────────────────────────────────────
import requests

GRAPH = "https://graph.microsoft.com/v1.0"


def graph_token() -> str:
    """App-only when GRAPH_CLIENT_SECRET is set, otherwise interactive device code."""
    import msal
    authority = f"https://login.microsoftonline.com/{GRAPH_TENANT_ID}"
    if not GRAPH_CLIENT_ID:
        raise RuntimeError("Set GRAPH_CLIENT_ID (app registration) to use the Graph path.")

    if GRAPH_CLIENT_SECRET:
        app = msal.ConfidentialClientApplication(
            GRAPH_CLIENT_ID, authority=authority, client_credential=GRAPH_CLIENT_SECRET)
        result = app.acquire_token_for_client(["https://graph.microsoft.com/.default"])
    else:
        pub = msal.PublicClientApplication(GRAPH_CLIENT_ID, authority=authority)
        scopes = ["OnlineMeetings.Read", "OnlineMeetingTranscript.Read.All"]
        accounts = pub.get_accounts()
        result = pub.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
        if not result:
            flow = pub.initiate_device_flow(scopes=scopes)
            print(flow["message"])
            result = pub.acquire_token_by_device_flow(flow)

    token = (result or {}).get("access_token")
    if not token:
        raise RuntimeError(f"Token acquisition failed: {result}")
    return token


def fetch_transcript_via_graph(recap: dict) -> str:
    """Return the newest VTT transcript for the meeting described by a recap link."""
    if not recap.get("thread_id") or not recap.get("organizer_id"):
        raise ValueError("Recap URL is missing threadId / organizerId.")

    headers = {"Authorization": f"Bearer {graph_token()}"}
    user = recap["organizer_id"]

    flt = f"chatInfo/threadId eq '{recap['thread_id']}'"
    r = requests.get(f"{GRAPH}/users/{user}/onlineMeetings",
                     params={"$filter": flt}, headers=headers, timeout=60)
    r.raise_for_status()
    meetings = r.json().get("value", [])
    if not meetings:
        raise LookupError("No online meeting matched that threadId.")
    meeting_id = meetings[0]["id"]

    r = requests.get(f"{GRAPH}/users/{user}/onlineMeetings/{meeting_id}/transcripts",
                     headers=headers, timeout=60)
    r.raise_for_status()
    transcripts = sorted(r.json().get("value", []),
                         key=lambda t: t.get("createdDateTime", ""), reverse=True)
    if not transcripts:
        raise LookupError("The meeting has no transcript (recording without transcription?).")

    r = requests.get(
        f"{GRAPH}/users/{user}/onlineMeetings/{meeting_id}/transcripts/{transcripts[0]['id']}/content",
        params={"$format": "text/vtt"}, headers=headers, timeout=120)
    r.raise_for_status()
    return r.text


print("Graph helpers ready. Set TRANSCRIPT_SOURCE = 'graph' to use them.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 ▸ Transcript loading & cleanup  ->  TRANSCRIPT_TEXT
# ─────────────────────────────────────────────────────────────────────────────
FILLER = re.compile(r"\b(um+|uh+|erm|you know|i mean|kind of|sort of|like,)\b", re.I)


def vtt_to_dialogue(vtt: str) -> str:
    """Collapse WebVTT cues into 'Speaker: sentence' lines, merging consecutive turns."""
    turns, speaker, buf = [], None, []
    for raw in vtt.splitlines():
        line = raw.strip()
        if (not line or line.upper().startswith(("WEBVTT", "NOTE", "STYLE"))
                or "-->" in line or line.isdigit()
                or re.fullmatch(r"[0-9a-f\-]{8,}(/\d+-\d+)?", line, re.I)):
            continue
        m = re.match(r"<v\s+([^>]+?)>(.*?)(?:</v>)?$", line, re.S)
        who, text = (m.group(1).strip(), m.group(2)) if m else (speaker, line)
        text = re.sub(r"<[^>]+>", "", text).strip()
        if not text:
            continue
        if who != speaker:
            if buf:
                turns.append(f"{speaker or 'Unknown'}: {' '.join(buf)}")
            speaker, buf = who, [text]
        else:
            buf.append(text)
    if buf:
        turns.append(f"{speaker or 'Unknown'}: {' '.join(buf)}")
    return "\n".join(turns)


def docx_to_text(path: Path) -> str:
    from docx import Document
    return "\n".join(p.text for p in Document(str(path)).paragraphs if p.text.strip())


def load_transcript() -> str:
    if TRANSCRIPT_SOURCE == "graph":
        return vtt_to_dialogue(fetch_transcript_via_graph(RECAP))
    if TRANSCRIPT_SOURCE == "text":
        return TRANSCRIPT_INLINE.strip()
    if TRANSCRIPT_SOURCE == "file":
        p = Path(TRANSCRIPT_FILE)
        if not p.exists():
            raise FileNotFoundError(
                f"{p.resolve()} not found. Download the transcript from the Teams recap "
                f"(Recap > Transcript > Download) or switch TRANSCRIPT_SOURCE to 'text'.")
        if p.suffix.lower() == ".vtt":
            return vtt_to_dialogue(p.read_text(encoding="utf-8-sig"))
        if p.suffix.lower() == ".docx":
            return docx_to_text(p)
        return p.read_text(encoding="utf-8-sig")
    raise ValueError(f"Unknown TRANSCRIPT_SOURCE: {TRANSCRIPT_SOURCE}")


def clean_transcript(text: str, redact: bool = False) -> str:
    text = FILLER.sub("", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if redact:
        alias, n = {}, 0
        out = []
        for line in text.splitlines():
            m = re.match(r"^([^:]{2,60}):\s*(.*)$", line)
            if m:
                who = m.group(1)
                if who not in alias:
                    n += 1
                    alias[who] = f"Speaker {n}"
                line = f"{alias[who]}: {m.group(2)}"
            out.append(line)
        text = "\n".join(out)
    return text


TRANSCRIPT_TEXT = clean_transcript(load_transcript(), redact=REDACT_NAMES)
(OUT_DIR / "transcript.txt").write_text(TRANSCRIPT_TEXT, encoding="utf-8")

words = len(TRANSCRIPT_TEXT.split())
print(f"Transcript loaded: {words:,} words / ~{words * 4 // 3:,} tokens")
print("-" * 78)
print(TRANSCRIPT_TEXT[:1200] + ("..." if len(TRANSCRIPT_TEXT) > 1200 else ""))


## Step 2 — Structured summary: discussion points, decisions, proposals

Long transcripts are map-reduced: each chunk is condensed first, then the condensed notes are
extracted into a single JSON object so downstream cells (and the diagram) work off strict schema
rather than prose.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 ▸ Azure OpenAI client + chunking helpers
# ─────────────────────────────────────────────────────────────────────────────
from openai import AzureOpenAI, BadRequestError

if not AOAI_ENDPOINT:
    raise RuntimeError("AZURE_OPENAI_ENDPOINT is not set.")

if AOAI_API_KEY:
    client = AzureOpenAI(azure_endpoint=AOAI_ENDPOINT, api_key=AOAI_API_KEY,
                         api_version=AOAI_API_VERSION)
else:
    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
    token_provider = get_bearer_token_provider(
        DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default")
    client = AzureOpenAI(azure_endpoint=AOAI_ENDPOINT, azure_ad_token_provider=token_provider,
                         api_version=AOAI_API_VERSION)

# Reasoning models (gpt-5.x, o1/o3/o4) reject a custom `temperature`.
SUPPORTS_TEMPERATURE = not re.match(r"^(gpt-5|o[134])", AOAI_DEPLOYMENT, re.I)


def ask(system: str, user: str, json_mode: bool = False, temperature: float = 0.1) -> str:
    kwargs = {"response_format": {"type": "json_object"}} if json_mode else {}
    if SUPPORTS_TEMPERATURE:
        kwargs["temperature"] = temperature
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        resp = client.chat.completions.create(model=AOAI_DEPLOYMENT, messages=messages, **kwargs)
    except BadRequestError:                       # model rejected an unsupported parameter
        kwargs.pop("temperature", None)
        resp = client.chat.completions.create(model=AOAI_DEPLOYMENT, messages=messages, **kwargs)
    return resp.choices[0].message.content


def ask_json(system: str, user: str, temperature: float = 0.1) -> dict:
    raw = ask(system, user, json_mode=True, temperature=temperature)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.S)          # tolerate stray prose / code fences
        if not m:
            raise
        return json.loads(m.group(0))


def chunk_text(text: str, max_chars: int = 24_000, overlap: int = 800) -> list[str]:
    """Split on line boundaries so a speaker turn is never cut in half."""
    if len(text) <= max_chars:
        return [text]
    chunks, buf, size = [], [], 0
    for line in text.splitlines(keepends=True):
        if size + len(line) > max_chars and buf:
            chunks.append("".join(buf))
            tail = "".join(buf)[-overlap:]
            buf, size = [tail, line], len(tail) + len(line)
        else:
            buf.append(line); size += len(line)
    if buf:
        chunks.append("".join(buf))
    return chunks


CHUNKS = chunk_text(TRANSCRIPT_TEXT)
print(f"Model: {AOAI_DEPLOYMENT}  |  transcript split into {len(CHUNKS)} chunk(s)")


ModuleNotFoundError: No module named 'openai'

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 ▸ Map-reduce extraction  ->  SUMMARY (strict JSON)
# ─────────────────────────────────────────────────────────────────────────────
CONDENSE_SYS = (
    "You are a technical meeting analyst. Condense the transcript excerpt into dense factual notes. "
    "Preserve named systems, products, data sources, integration patterns, numbers, owners and dates. "
    "Keep every decision, proposal and objection. Never invent facts. Output plain bullet points."
)

SUMMARY_SYS = f"""You are a technical meeting analyst. From the notes, produce ONE JSON object:

{{
  "meeting": {{"title": str, "date": str, "purpose": str, "attendees": [str]}},
  "executive_summary": str,
  "summary_points": [{{"topic": str, "detail": str}}],
  "decisions": [{{"decision": str, "rationale": str, "owner": str, "status": "agreed|tentative|deferred"}}],
  "proposals": [{{"proposal": str, "proposed_by": str, "benefit": str, "open_questions": [str]}}],
  "action_items": [{{"action": str, "owner": str, "due": str}}],
  "risks": [{{"risk": str, "impact": str, "mitigation": str}}],
  "open_questions": [str]
}}

Rules:
- Only use information present in the notes. Use "not stated" for unknown scalars, [] for unknown lists.
- A decision is something the group settled on; a proposal is something suggested but not settled.
- Keep each string under 40 words. Return JSON only.
Meeting title: {MEETING_TITLE}. Meeting date: {MEETING_DATE}."""

if len(CHUNKS) == 1:
    condensed = CHUNKS[0]
else:
    parts = []
    for i, c in enumerate(CHUNKS, 1):
        print(f"  condensing chunk {i}/{len(CHUNKS)} ...")
        parts.append(ask(CONDENSE_SYS, f"Excerpt {i} of {len(CHUNKS)}:\n\n{c}"))
    condensed = "\n\n".join(parts)

SUMMARY = ask_json(SUMMARY_SYS, f"NOTES:\n\n{condensed}")
(OUT_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=2), encoding="utf-8")

print(f"\nExtracted: {len(SUMMARY.get('summary_points', []))} points | "
      f"{len(SUMMARY.get('decisions', []))} decisions | "
      f"{len(SUMMARY.get('proposals', []))} proposals | "
      f"{len(SUMMARY.get('action_items', []))} actions")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 ▸ Render the summary as Markdown (displayed + saved)
# ─────────────────────────────────────────────────────────────────────────────
from IPython.display import Markdown, display

def _g(d, k, default="not stated"):
    v = d.get(k)
    return v if v not in (None, "", []) else default

def _table(rows, headers, keys):
    if not rows:
        return "_None captured._\n"
    md = "| " + " | ".join(headers) + " |\n"
    md += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    for r in rows:
        cells = [str(_g(r, k)).replace("|", "\\|").replace("\n", " ") for k in keys]
        md += "| " + " | ".join(cells) + " |\n"
    return md + "\n"

m = SUMMARY.get("meeting", {})
md = [f"# {_g(m, 'title', MEETING_TITLE)}",
      f"**Date:** {_g(m, 'date', MEETING_DATE)}  ",
      f"**Attendees:** {', '.join(m.get('attendees') or ['not stated'])}  ",
      f"**Purpose:** {_g(m, 'purpose')}",
      "\n## Executive summary\n",
      _g(SUMMARY, "executive_summary"),
      "\n## Discussion points\n"]

for p in SUMMARY.get("summary_points", []):
    md.append(f"- **{_g(p, 'topic')}** — {_g(p, 'detail')}")

md += ["\n## Decisions\n",
       _table(SUMMARY.get("decisions", []), ["Decision", "Rationale", "Owner", "Status"],
              ["decision", "rationale", "owner", "status"]),
       "## Proposals\n",
       _table(SUMMARY.get("proposals", []), ["Proposal", "Proposed by", "Benefit"],
              ["proposal", "proposed_by", "benefit"]),
       "## Action items\n",
       _table(SUMMARY.get("action_items", []), ["Action", "Owner", "Due"],
              ["action", "owner", "due"]),
       "## Risks\n",
       _table(SUMMARY.get("risks", []), ["Risk", "Impact", "Mitigation"],
              ["risk", "impact", "mitigation"]),
       "## Open questions\n"]
md += [f"- {q}" for q in SUMMARY.get("open_questions", [])] or ["_None captured._"]
md.append("\n---\n_Generated by **Meeting Notes to Architecture**. AI-generated content — verify before circulating._")

SUMMARY_MD = "\n".join(md)
(OUT_DIR / "summary.md").write_text(SUMMARY_MD, encoding="utf-8")
display(Markdown(SUMMARY_MD))


## Step 3 — Architecture diagram

The model converts the discussion into an explicit architecture spec (layered nodes + typed flows),
which is then rendered by Graphviz at **300 DPI PNG + vector SVG**. Nothing is hallucinated into the
picture: components not named in the meeting are marked `"proposed": true` and drawn with a dashed
border so reviewers can tell agreed scope from suggestion.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 ▸ Derive the architecture spec  ->  ARCH (strict JSON)
# ─────────────────────────────────────────────────────────────────────────────
ARCH_SYS = """You are a solution architect. From the meeting notes, derive the architecture that was
discussed and return ONE JSON object:

{
  "title": str,
  "layers": [{"id": str, "label": str, "order": int}],
  "components": [{"id": str, "label": str, "technology": str, "layer": str,
                  "proposed": bool, "note": str}],
  "flows": [{"from": str, "to": str, "label": str,
             "type": "batch|streaming|api|governance|ai|user"}],
  "assumptions": [str]
}

Rules:
- Layers describe the flow left-to-right, e.g. Sources -> Ingestion -> Storage -> Transformation ->
  Serving -> Consumption, plus a "Governance & Security" layer when relevant. Order starts at 1.
- `id` values are lowercase snake_case and unique. Every flow endpoint MUST match a component id.
- `layer` MUST match a layer id.
- `proposed` is true when the component was only suggested, not agreed.
- `technology` is the concrete product named in the meeting (e.g. "Fabric Lakehouse", "Azure Data
  Factory"); use "" when no product was named.
- Include only components discussed in the meeting. 8-25 components. Labels under 6 words.
- Return JSON only."""

ARCH = ask_json(ARCH_SYS,
                "MEETING SUMMARY (JSON):\n" + json.dumps(SUMMARY, indent=2) +
                "\n\nDETAILED NOTES:\n" + condensed)

# ── Validate & repair references so rendering never fails ─────────────────────
layer_ids = {l["id"] for l in ARCH.get("layers", [])}
if not layer_ids:
    ARCH["layers"] = [{"id": "solution", "label": "Solution", "order": 1}]
    layer_ids = {"solution"}
    for c in ARCH.get("components", []):
        c["layer"] = "solution"

comp_ids = {c["id"] for c in ARCH.get("components", [])}
for c in ARCH.get("components", []):
    if c.get("layer") not in layer_ids:
        c["layer"] = sorted(ARCH["layers"], key=lambda l: l.get("order", 0))[0]["id"]

dropped = [f for f in ARCH.get("flows", []) if f.get("from") not in comp_ids or f.get("to") not in comp_ids]
ARCH["flows"] = [f for f in ARCH.get("flows", []) if f not in dropped]

(OUT_DIR / "architecture.json").write_text(json.dumps(ARCH, indent=2), encoding="utf-8")

print(f"{ARCH.get('title', 'Architecture')}")
print(f"  layers     : {len(ARCH['layers'])}")
print(f"  components : {len(comp_ids)}  ({sum(1 for c in ARCH['components'] if c.get('proposed'))} proposed)")
print(f"  flows      : {len(ARCH['flows'])}" + (f"  ({len(dropped)} dangling flow(s) dropped)" if dropped else ""))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 ▸ Render high-resolution architecture diagram (PNG @ 300 DPI + SVG)
# ─────────────────────────────────────────────────────────────────────────────
import html as _html

PALETTE = ["#E3F2FD", "#E8F5E9", "#FFF8E1", "#F3E5F5", "#FFEBEE", "#E0F7FA", "#F1F8E9", "#FCE4EC"]
BORDER  = ["#1565C0", "#2E7D32", "#F9A825", "#6A1B9A", "#C62828", "#00838F", "#558B2F", "#AD1457"]
EDGE_STYLE = {
    "batch":      {"color": "#1565C0", "style": "solid"},
    "streaming":  {"color": "#00838F", "style": "solid", "penwidth": "2.4"},
    "api":        {"color": "#6A1B9A", "style": "solid"},
    "ai":         {"color": "#AD1457", "style": "solid"},
    "governance": {"color": "#616161", "style": "dashed"},
    "user":       {"color": "#2E7D32", "style": "solid"},
}


def node_label(c: dict) -> str:
    label = _html.escape(c.get("label", c["id"]))
    tech  = _html.escape(c.get("technology") or "")
    tech_row = f'<TR><TD><FONT POINT-SIZE="9" COLOR="#455A64">{tech}</FONT></TD></TR>' if tech else ""
    tag_row  = ('<TR><TD><FONT POINT-SIZE="8" COLOR="#B71C1C"><I>proposed</I></FONT></TD></TR>'
                if c.get("proposed") else "")
    return (f'<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0">'
            f'<TR><TD><B>{label}</B></TD></TR>{tech_row}{tag_row}</TABLE>>')


def build_graph(arch: dict):
    import graphviz
    g = graphviz.Digraph("architecture", format="png")
    g.attr(rankdir="LR", splines="ortho", nodesep="0.45", ranksep="1.0",
           bgcolor="white", dpi=str(DIAGRAM_DPI), pad="0.4",
           fontname="Segoe UI", labelloc="t", fontsize="22",
           label=f"\n{arch.get('title', MEETING_TITLE)}\n")
    g.attr("node", shape="box", style="rounded,filled", fontname="Segoe UI",
           fontsize="12", margin="0.18,0.11", penwidth="1.6")
    g.attr("edge", fontname="Segoe UI", fontsize="9", arrowsize="0.7", penwidth="1.4")

    layers = sorted(arch["layers"], key=lambda l: l.get("order", 0))
    for i, layer in enumerate(layers):
        fill, line = PALETTE[i % len(PALETTE)], BORDER[i % len(BORDER)]
        with g.subgraph(name=f"cluster_{layer['id']}") as sub:
            sub.attr(label=layer.get("label", layer["id"]), style="rounded,filled",
                     color=line, fillcolor=fill + "60", fontname="Segoe UI Semibold",
                     fontsize="13", fontcolor=line, penwidth="1.8", margin="16")
            for c in arch["components"]:
                if c.get("layer") == layer["id"]:
                    sub.node(c["id"], label=node_label(c), fillcolor=fill, color=line,
                             style="rounded,filled,dashed" if c.get("proposed") else "rounded,filled",
                             tooltip=c.get("note", ""))

    for f in arch["flows"]:
        style = EDGE_STYLE.get(f.get("type", "batch"), EDGE_STYLE["batch"])
        g.edge(f["from"], f["to"], label=" " + _html.escape(f.get("label", "")) + " ", **style)
    return g


DIAGRAM_PNG = DIAGRAM_SVG = None
try:
    graph = build_graph(ARCH)
    base = str(OUT_DIR / "architecture")
    DIAGRAM_PNG = graph.render(filename=base, format="png", cleanup=True)
    DIAGRAM_SVG = graph.render(filename=base, format="svg", cleanup=True)
    (OUT_DIR / "architecture.gv").write_text(graph.source, encoding="utf-8")
    print(f"PNG @ {DIAGRAM_DPI} DPI : {DIAGRAM_PNG}")
    print(f"SVG (vector)   : {DIAGRAM_SVG}")
except Exception as e:
    print(f"Graphviz render skipped: {type(e).__name__}: {e}")
    print("Install the binaries (winget install Graphviz.Graphviz) or use the Mermaid output below.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 ▸ Preview the diagram inline (vector SVG scales without blurring)
# ─────────────────────────────────────────────────────────────────────────────
from IPython.display import SVG, Image

if DIAGRAM_SVG:
    display(SVG(filename=DIAGRAM_SVG))
elif DIAGRAM_PNG:
    display(Image(filename=DIAGRAM_PNG, width=1100))
else:
    print("No rendered diagram — see the Mermaid fallback in the next cell.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 ▸ Mermaid export (portable — paste into Loop, ADO wiki, GitHub, docs)
# ─────────────────────────────────────────────────────────────────────────────
def to_mermaid(arch: dict) -> str:
    esc = lambda s: str(s).replace('"', "'").replace("\n", " ")
    lines = ["flowchart LR"]
    for layer in sorted(arch["layers"], key=lambda l: l.get("order", 0)):
        lines.append(f'  subgraph {layer["id"]}["{esc(layer.get("label", layer["id"]))}"]')
        lines.append("    direction TB")
        for c in arch["components"]:
            if c.get("layer") == layer["id"]:
                tech = f"<br/><small>{esc(c['technology'])}</small>" if c.get("technology") else ""
                lines.append(f'    {c["id"]}["{esc(c.get("label", c["id"]))}{tech}"]')
        lines.append("  end")
    for f in arch["flows"]:
        arrow = "-.->" if f.get("type") == "governance" else "-->"
        label = esc(f.get("label", ""))
        lines.append(f'  {f["from"]} {arrow}{f"|{label}|" if label else ""} {f["to"]}')
    for c in arch["components"]:
        if c.get("proposed"):
            lines.append(f'  style {c["id"]} stroke-dasharray: 5 5,stroke:#B71C1C')
    return "\n".join(lines)


MERMAID = to_mermaid(ARCH)
(OUT_DIR / "architecture.mmd").write_text(MERMAID, encoding="utf-8")
display(Markdown(f"```mermaid\n{MERMAID}\n```"))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 ▸ Assemble the shareable report + list artifacts
# ─────────────────────────────────────────────────────────────────────────────
report = [SUMMARY_MD.rsplit("\n---\n", 1)[0],
          "\n## Architecture discussed\n"]
if DIAGRAM_PNG:
    report.append(f"![Architecture]({Path(DIAGRAM_PNG).name})\n")
report.append(f"```mermaid\n{MERMAID}\n```\n")
if ARCH.get("assumptions"):
    report.append("**Assumptions made while drawing this diagram:**\n")
    report += [f"- {a}" for a in ARCH["assumptions"]]
report.append("\n---\n_AI-generated from the meeting transcript. Verify decisions and owners "
              "with attendees before treating this as a record._")

REPORT_MD = "\n".join(report)
(OUT_DIR / "report.md").write_text(REPORT_MD, encoding="utf-8")

print("Artifacts:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.stat().st_size / 1024:8.1f} KB  {f.name}")
print(f"\nFolder: {OUT_DIR.resolve()}")
